In [1]:

%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')

from verimon.utils import setup_logging

setup_logging()

In [15]:
from verimon import loaders
from random import randrange
from math import sqrt

gb_e = "../tests/gb_exact.pm"
gb = loaders.load_dfa(gb_e)

mc_sl_u_nxn = "../tests/snake_ladder/mc_u_nxn.pm"

def random_ladder(n):
    source = randrange(1,n-int(sqrt(n)))
    dest = randrange(source+int(sqrt(n)), int(min(n, source + n / 2)))
    return source, dest

def random_snake(n):
    source = randrange(int(sqrt(n))+2,n)
    dest = randrange(1, source-int(sqrt(n)))
    return source, dest

n = 5**2
ladders = {0:0}
snakes = {0:0}
while not set(ladders.keys()).isdisjoint(set(snakes.keys())):
    ladders = dict(random_ladder(n) for _ in range(int(sqrt(n))))
    snakes = dict(random_snake(n) for _ in range(int(sqrt(n))))

# Random snakes and ladders
mc = loaders.load_snl_stormpy(mc_sl_u_nxn, n, ladders, snakes)

milton_snakes = {98: 76, 95: 75, 93: 73, 87: 24, 64: 60, 62: 19, 55: 53, 49: 11, 47: 26, 16: 6}
milton_ladders = {1: 38, 4: 14, 9: 31, 28: 64, 40: 42, 36: 44, 51: 67, 71: 91, 80: 100}
# mc = loaders.load_snl(mc_sl_u_nxn, n := 10**2, ladders:=milton_ladders, snakes:=milton_snakes)

In [16]:

from stormvogel.mapping import stormpy_to_stormvogel

mc_sv = stormpy_to_stormvogel(mc)
loaders._add_valuation_to_sv_labels(mc, mc_sv)
# show(mc_sv)

In [17]:
%matplotlib notebook
from verimon.draw import animate_player_movement
import math
from IPython.display import HTML

player_path = [(0, [])]

goal_squares = [next(int(l[5:-1]) for l in state.labels if l.startswith("[pos")) 
                for state in mc_sv.states.values() 
                if "good" in state.labels]

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

<IPython.core.display.Javascript object>

In [ ]:
from aalpy import run_Lstar
from verimon.MonitorLearning import FilteringSUL, VerimonEqOracle

setup_logging()


threshold = 0.4
slack = 0.1
horizon = 12
relative_error = 0.1

alphabet = ["init", "normal", "snake", "ladder"]

filtering_sul = FilteringSUL(
    mc, 
    "init", 
    alphabet, 
    'P>0.9 [ "good" ]', 
    threshold, 
)
eq_oracle = VerimonEqOracle(
    alphabet,
    filtering_sul,
    mc_sv,
    gb,
    threshold,
    slack,
    horizon,
    'P>0.9 [ "good" ]',
    'good',
    relative_error
    
)
learned_monitor = run_Lstar(
    alphabet,
    filtering_sul,
    eq_oracle,
    automaton_type="dfa",
    print_level=2,
)

Hypothesis 1: 1 states.
Visualization started in the background thread.
DEBUG:2024-10-18 14:27:11,172 - (0.00s) - MonitorLearning.py - Finding false negative probability 
DEBUG:2024-10-18 14:27:11,173 - (0.00s) - verify.py - Building model 
DEBUG:2024-10-18 14:27:11,174 - (0.00s) - verify.py - Unrolling done 
DEBUG:2024-10-18 14:27:11,178 - (0.00s) - verify.py - Pruning done 
INFO:2024-10-18 14:27:11,189 - (0.01s) - MDP_product.py - New good states become: [['[pos=25]']] 
DEBUG:2024-10-18 14:27:11,190 - (0.00s) - verify.py - Apply spec done 
DEBUG:2024-10-18 14:27:11,191 - (0.00s) - MDP_product.py - Finished product setup 
DEBUG:2024-10-18 14:27:11,209 - (0.02s) - MDP_product.py - Created product states 
DEBUG:2024-10-18 14:27:11,222 - (0.01s) - MDP_product.py - Created product transitions 
DEBUG:2024-10-18 14:27:11,223 - (0.00s) - verify.py - creating product done 
Model saved to model1.pdf.
DEBUG:2024-10-18 14:27:11,224 - (0.00s) - verify.py - Creating trace 
> progress 26.079%, elap

In [36]:
from verimon.MonitorLearning import aalpy_dfa_to_stormvogel
from verimon.verify import *

mon_cycl = aalpy_dfa_to_stormvogel(learned_monitor)
result_goal, trace, assignment, product = false_positive(mc_sv, mon_cycl, gb, 11)

DEBUG:2024-10-18 12:14:18,440 - (15.53s) - verify.py - Building model 
DEBUG:2024-10-18 12:14:18,442 - (0.00s) - verify.py - Unrolling done 
DEBUG:2024-10-18 12:14:18,452 - (0.01s) - verify.py - Pruning done 
INFO:2024-10-18 12:14:18,459 - (0.01s) - MDP_product.py - New good states become: [['[pos=36]']] 
DEBUG:2024-10-18 12:14:18,459 - (0.00s) - verify.py - Apply spec done 
DEBUG:2024-10-18 12:14:18,460 - (0.00s) - MDP_product.py - Finished product setup 
DEBUG:2024-10-18 12:14:18,809 - (0.35s) - MDP_product.py - Created product states 
DEBUG:2024-10-18 12:14:18,861 - (0.05s) - MDP_product.py - Created product transitions 
DEBUG:2024-10-18 12:14:18,862 - (0.00s) - verify.py - creating product done 
DEBUG:2024-10-18 12:14:18,862 - (0.00s) - verify.py - Creating trace 
> progress 0.001%, elapsed 3 s, estimated 260098 s (3 days), iters = {MDP: 105}, opt = 0.7269
> progress 0.002%, elapsed 6 s, estimated 258910 s (2 days), iters = {MDP: 219}, opt = 0.7269
> progress 0.003%, elapsed 9 s, e

In [37]:
%matplotlib notebook
from verimon.draw import animate_player_movement
import math
from IPython.display import HTML

# player_path = [(0, [])]
poss = product.simulate_paynt_assignment(assignment, 1000)
player_path = poss

goal_squares = [next(int(l[5:-1]) for l in state.labels if l.startswith("[pos")) 
                for state in product.mc.states.values() 
                if "good" in state.labels]

animation = animate_player_movement(int(math.sqrt(n)), snakes, ladders, goal_squares, player_path)
HTML(animation.to_jshtml())

s0, labels=[pos=0] !g0 !s0 !l0 init

	--[0, 1:normal]-->
s77, labels=!g0 !l1 !s1 [pos=1] normal
	--[1, 1:normal]-->
s229, labels=!g0 !l3 !s5 [pos=5] normal
	--[2, 1:normal]-->
s453, labels=!g0 !l6 !s7 [pos=7] normal
	--[3, 1:normal]-->
s751, labels=!g0 !l10 !s9 [pos=9] normal
	--[4, 1:normal]-->
s1127, labels=!g0 !l15 !s15 [pos=14] normal
	--[5, 1:normal]-->
s1574, labels=!g0 !l21 !s18 [pos=18] normal
	--[6, 1:normal]-->
s2098, labels=!g0 !l28 !s24 [pos=23] normal
	--[7, 1:normal]-->
s2621, labels=!g0 !l35 !s29 [pos=28] normal
	--[9, 1:normal]-->
s3068, labels=!g0 !l41 !s32 [pos=33] normal
	--[11, 1:normal]-->
s3439, labels=!g0 !l46 !s33 [pos=35] accepting horizon normal
	--[13, 4:end]-->
s2, labels=stop
it took 5 tries until the goal was reached


<IPython.core.display.Javascript object>